# 01 - Exploracao de Dados

Objetivo deste notebook:
- localizar o arquivo bruto de IDH em `datasets/idh`
- ler o arquivo corretamente
- entender a estrutura da base
- avaliar colunas, tipos, nulos e duplicatas
- inspecionar a granularidade municipal
- levantar as regras que serao aplicadas no notebook 02


## 1. Imports e configuracao

In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PASTA_IDH = Path('../datasets/idh')
ANO_REFERENCIA = 2010

## 2. Localizar os arquivos disponiveis

In [3]:
arquivos_idh = sorted([arquivo for arquivo in PASTA_IDH.iterdir() if arquivo.is_file()])
arquivos_idh

[PosixPath('../datasets/idh/data.csv')]

## 3. Selecionar o arquivo bruto

Observacao: o arquivo atual veio com extensao `.csv`, mas pela inspecao inicial ele e uma planilha Excel.
Por isso, a leitura sera feita com `read_excel`.

In [4]:
arquivo_idh = arquivos_idh[0]
arquivo_idh

PosixPath('../datasets/idh/data.csv')

## 4. Ler a planilha e inspecionar as abas

In [5]:
planilha = pd.ExcelFile(arquivo_idh)
planilha.sheet_names

['Worksheet']

In [6]:
df_idh_raw = pd.read_excel(arquivo_idh, sheet_name=planilha.sheet_names[0])
df_idh_raw.head()

,Territorialidade,Posição IDHM,IDHM,Posição IDHM Renda,IDHM Renda,Posição IDHM Educação,IDHM Educação,Posição IDHM Longevidade,IDHM Longevidade
0,São Caetano do Sul (SP),1,0.862,1,0.891,2,0.811,19,0.887
1,Águas de São Pedro (SP),2,0.854,12,0.849,1,0.825,11,0.890
2,Florianópolis (SC),3,0.847,5,0.870,5,0.800,147,0.873
3,Vitória (ES),4,0.845,3,0.876,4,0.805,551,0.855
4,Balneário Camboriú (SC),4,0.845,10,0.854,6,0.789,1,0.894


## 5. Visao geral da base

In [7]:
print(f'Quantidade de linhas: {df_idh_raw.shape[0]}')
print(f'Quantidade de colunas: {df_idh_raw.shape[1]}')
print('\nColunas encontradas:')
for coluna in df_idh_raw.columns:
    print('-', coluna)

Quantidade de linhas: 5565
Quantidade de colunas: 9

Colunas encontradas:
- Territorialidade
- Posição IDHM
- IDHM
- Posição IDHM Renda
- IDHM Renda
- Posição IDHM Educação
- IDHM Educação
- Posição IDHM Longevidade
- IDHM Longevidade


In [8]:
df_idh_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5565 entries, 0 to 5564
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Territorialidade          5565 non-null   object 
 1   Posição IDHM              5565 non-null   int64  
 2   IDHM                      5565 non-null   float64
 3   Posição IDHM Renda        5565 non-null   int64  
 4   IDHM Renda                5565 non-null   float64
 5   Posição IDHM Educação     5565 non-null   int64  
 6   IDHM Educação             5565 non-null   float64
 7   Posição IDHM Longevidade  5565 non-null   int64  
 8   IDHM Longevidade          5565 non-null   float64
dtypes: float64(4), int64(4), object(1)
memory usage: 391.4+ KB


## 6. Estatisticas iniciais de qualidade

In [9]:
nulos_por_coluna = df_idh_raw.isna().sum().sort_values(ascending=False)
display(nulos_por_coluna.to_frame('qtd_nulos'))

,qtd_nulos
Territorialidade,0
Posição IDHM,0
IDHM,0
Posição IDHM Renda,0
IDHM Renda,0
Posição IDHM Educação,0
IDHM Educação,0
Posição IDHM Longevidade,0
IDHM Longevidade,0


In [10]:
duplicatas_exatas = df_idh_raw.duplicated().sum()
print(f'Duplicatas exatas: {duplicatas_exatas}')

Duplicatas exatas: 0


In [11]:
display(df_idh_raw.describe(include='all').transpose())

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Territorialidade,5565,5565,São Caetano do Sul (SP),1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Posição IDHM,5565.0,NaN,NaN,NaN,2772.268104,1607.377806,1.0,1362.0,2776.0,4167.0,5565.0
IDHM,5565.0,NaN,NaN,NaN,0.659157,0.071997,0.418,0.599,0.665,0.718,0.862
Posição IDHM Renda,5565.0,NaN,NaN,NaN,2773.005571,1607.262066,1.0,1372.0,2767.0,4161.0,5565.0
IDHM Renda,5565.0,NaN,NaN,NaN,0.642873,0.080662,0.4,0.572,0.654,0.707,0.891
Posição IDHM Educação,5565.0,NaN,NaN,NaN,2774.874753,1606.714793,1.0,1370.0,2776.0,4163.0,5565.0
IDHM Educação,5565.0,NaN,NaN,NaN,0.559094,0.093328,0.207,0.49,0.56,0.631,0.825
Posição IDHM Longevidade,5565.0,NaN,NaN,NaN,2764.980773,1609.310644,1.0,1370.0,2754.0,4168.0,5564.0
IDHM Longevidade,5565.0,NaN,NaN,NaN,0.801564,0.044681,0.672,0.769,0.808,0.836,0.894


## 7. Inspecao da coluna Territorialidade

Esta coluna e critica porque dela devem sair:
- `nome_municipio`
- `sigla_uf`
- e mais tarde o vinculo com `codigo_municipio`(nesse caso do IDH que não veio) 

In [12]:
df_idh_raw['Territorialidade'].head(20)

0      São Caetano do Sul (SP)
1      Águas de São Pedro (SP)
2           Florianópolis (SC)
3                 Vitória (ES)
4      Balneário Camboriú (SC)
5                  Santos (SP)
6                 Niterói (RJ)
7                 Joaçaba (SC)
8                Brasília (DF)
9                Curitiba (PR)
10                Jundiaí (SP)
11               Valinhos (SP)
12                Vinhedo (SP)
13             Araraquara (SP)
14            Santo André (SP)
15    Santana de Parnaíba (SP)
16              Nova Lima (MG)
17          Ilha Solteira (SP)
18              Americana (SP)
19         Belo Horizonte (MG)
Name: Territorialidade, dtype: object

In [13]:
amostra_territorialidade = pd.DataFrame({
    'territorialidade': df_idh_raw['Territorialidade'],
    'nome_municipio_extraido': df_idh_raw['Territorialidade'].astype(str).str.replace(r'\s*\([A-Z]{2}\)$', '', regex=True),
    'sigla_uf_extraida': df_idh_raw['Territorialidade'].astype(str).str.extract(r'\(([A-Z]{2})\)$')[0],
})

display(amostra_territorialidade.head(20))

,territorialidade,nome_municipio_extraido,sigla_uf_extraida
0,São Caetano do Sul (SP),São Caetano do Sul,SP
1,Águas de São Pedro (SP),Águas de São Pedro,SP
2,Florianópolis (SC),Florianópolis,SC
3,Vitória (ES),Vitória,ES
4,Balneário Camboriú (SC),Balneário Camboriú,SC
5,Santos (SP),Santos,SP
6,Niterói (RJ),Niterói,RJ
7,Joaçaba (SC),Joaçaba,SC
8,Brasília (DF),Brasília,DF
9,Curitiba (PR),Curitiba,PR


## 8. Verificacao de chaves potenciais

Neste momento, como ainda nao temos `codigo_municipio`, a chave observada da base e:
- `Territorialidade`
- `ano` (fixo em 2010)

Mais adiante, no notebook 02, vamos transformar isso em:
- `codigo_municipio`
- `nome_municipio`
- `sigla_uf`
- `ano`


In [14]:
df_chave_temporaria = df_idh_raw.copy()
df_chave_temporaria['ano'] = ANO_REFERENCIA

duplicatas_chave_temporaria = df_chave_temporaria.duplicated(subset=['Territorialidade', 'ano']).sum()
print(f'Duplicatas na chave temporaria Territorialidade + ano: {duplicatas_chave_temporaria}')

Duplicatas na chave temporaria Territorialidade + ano: 0


## 9. Regras preliminares para o notebook 02

Com base na exploracao, a limpeza e padronizacao devem seguir estas etapas:

1. Ler o arquivo com `read_excel`
2. Criar a coluna `ano = 2010`
3. Extrair `nome_municipio` e `sigla_uf` a partir de `Territorialidade`
4. Converter colunas numericas para tipo numerico
5. Remover duplicatas, se existirem
6. Criar ou enriquecer a coluna `codigo_municipio`(código 7 digitos IBGE) por cruzamento com base auxiliar
7. Preparar a base final para integracao futura com crime, populacao e educacao.


## 10. Conclusao esperada desta etapa

Ao final deste notebook, precisamos confirmar:
- a estrutura real do arquivo
- a granularidade municipal
- a presenca dos indicadores de IDHM
- a viabilidade de extrair `nome_municipio` e `sigla_uf`
- os cuidados necessarios para criar `codigo_municipio` no notebook 02
